# PCA - Projet Final


## 1. Chargement des donnees


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

RANDOM_STATE = 42
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / 'data' / 'city_lifestyle_dataset.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de localiser le dossier projet contenant data/city_lifestyle_dataset.csv")

DATA_PATH = PROJECT_ROOT / 'data' / 'city_lifestyle_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()


## 2. Preprocessing


In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
X = df[numeric_cols].copy()

if X.isna().any().any():
    raise ValueError('Valeurs manquantes detectees dans les colonnes numeriques. Nettoyer avant PCA.')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Variables numeriques: {len(numeric_cols)}')
print(f'Shape de X: {X.shape}')


## 3. PCA 2D et visualisation


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
if 'city_name' in df.columns:
    pca_df['city_name'] = df['city_name'].values

print('Variance expliquee:')
for i, r in enumerate(pca.explained_variance_ratio_, start=1):
    print(f'PC{i}: {r:.4f}')
print(f'Variance cumulee (2 composantes): {pca.explained_variance_ratio_.sum():.4f}')

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.75, s=35, edgecolor='none')
ax.set_title('Projection PCA en 2D (PC1 vs PC2)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()


## 4. Observation
La projection 2D montre une structure continue avec des zones de concentration et quelques points plus eloignes.


## 5. Export 2D


In [ ]:
pca_output_path = OUTPUT_DIR / 'pca_2d.csv'
pca_df.to_csv(pca_output_path, index=False)
print(f'Export termine: {pca_output_path.resolve()}')
